In [ ]:
import numpy as np

with open('GUTENBERG.txt', 'r', encoding = "utf8") as fp:
  text = fp.read()
start_idx = text.find("THE MYSTERIOUS ISLAND")
end_idx = text.find("END OF THE PROJECT GUTENBERG EBOOK")
text = text[start_idx:end_idx]
char_set = set(text)
print("Total Length:", len(text))
print("Unique Characters:", len(char_set))
print("Unique Characters:", char_set)

In [ ]:
char_sorted = sorted(char_set)
print(char_sorted)
char2int = {c:i for i,c in enumerate(char_sorted)}
print(char2int)
char_array = np.array(char_sorted)
print(char_array)
text_encoded = np.array([char2int[c] for c in text], dtype = np.int32)
print(text_encoded)
#

print(text[:15], '== Encoding ==>', text_encoded[:15])

for ex in text_encoded[:5]:
  print('{} -> {}'.format(ex, char_array[ex]))


  

In [ ]:
import torch
from torch.utils.data import Dataset
seq_len = 40
chunk_size = seq_len + 1
text_chunks = [text_encoded[i:i+chunk_size] for i in range(len(text_encoded)-chunk_size)]
print(len(text_chunks))
# print(text_chunks[111])

class TextDataset(Dataset):
  
    def __init__(self, text_chunks):
        self.text_chunks = text_chunks

    def __len__(self):

        return len(self.text_chunks)

    def __getitem__(self, idx):

        text_chunk = self.text_chunks[idx]
        return text_chunk[:-1].long(), text_chunk[1:].long()

seq_dataset = TextDataset(torch.tensor(text_chunks))

In [ ]:
for i, (seq, target) in enumerate(seq_dataset):
    print(' Input (x): ', repr(''.join(char_array[seq])))
    print('Target (y): ', repr(''.join(char_array[target])))
    print()
    if i == 1:
        break

In [ ]:
from torch.utils.data import DataLoader
batch_size = 64
torch.manual_seed(1)
seq_dl = DataLoader(seq_dataset, batch_size=batch_size, shuffle=True, drop_last=True)

# Building a character-level RNN model

In [ ]:
import torch.nn as nn
class CharRNN(nn.Module):
  def __init__(self, vocab_size, embed_dim, rnn_hidden_state) :
    super().__init__()
    self.embedding = nn.Embedding(vocab_size, embed_dim)
    self.rnn_hidden_state = rnn_hidden_state
    self.rnn = nn.LSTM(embed_dim, rnn_hidden_state, batch_first=True)
    self.fc = nn.Linear(rnn_hidden_state, vocab_size)

  def forward(self, x, hidden, cell):
    out = self.embedding(x).unsqueeze(1)
    out, (hidden, cell) = self.rnn(out, (hidden, cell))
    out = self.fc(out).reshape(out.size(0), -1)
    return out, hidden, cell

  def init_state(self, batch_size):
    hidden = torch.zeros(1, batch_size, self.rnn_hidden_state)
    cell = torch.zeros(1, batch_size, self.rnn_hidden_state)
    return hidden, cell
#

In [ ]:
vocab_size = len(char_array)
embed_dim = 256
rnn_hidden_state = 512
torch.manual_seed(1)
model = CharRNN(vocab_size, embed_dim, rnn_hidden_state)
model

In [ ]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr = 0.001)
#

In [ ]:
num_epochs = 10000
torch.manual_seed(1)
for epoch in range(num_epochs):
  hidden, cell = model.init_state(batch_size)
  seq_batch, target_batch = next(iter(seq_dl))
  optimizer.zero_grad()
  loss = 0
  for c in range(seq_len):
    pred, hidden, cell = model(seq_batch[:, c], hidden, cell)
    loss += loss_fn(pred, target_batch[:, c])
  loss.backward()
  optimizer.step()
  loss = loss.item() / seq_len
  
  if (epoch+1) % 500 == 0:
    print(f'Epoch: {epoch}, Loss: {loss.item()}')
#